In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [19]:
load_dotenv()  # Load environment variables from .env file

True

In [20]:
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

In [21]:
def chat_with_model(messages : list[dict],model : str ="deepseek-flash") -> str:
    """
    Function to chat with the model using the provided messages and model name.

    Args:
        messages (list[dict]): A list of dictionaries containing the chat messages.
        model (str): The name of the model to use for chatting. Default is "deepseek-flash".

    Returns:
        str: The response from the model.
    """
    response = client.chat.completions.create(
    model="deepseek-flash",
    messages=messages
)

    return response.choices[0].message.content

In [22]:
zero_shot_messages = [
    {
        "role": "system",
        "content": (
            "You are a banking transaction classifier. "
            "Classify each transaction into one of these categories: "
            "Groceries, Utilities, EMI/Loan Repayment, ATM Withdrawal, "
            "Online Shopping, Food & Dining, Travel, Medical, Entertainment, or Other."
        )
    },
    {
        "role": "user",
        "content": (
            "Classify the following bank transactions:\n\n"
            "1. UPI/SWIGGY/ORDER123 — ₹450\n"
            "2. NEFT/HDFC HOME LOAN EMI — ₹23,000\n"
            "3. POS/BIGBASKET SUPERMART — ₹1,820\n"
            "4. ATM/ICICI BANK/MG ROAD — ₹5,000\n"
            "5. IMPS/IRCTC TICKET BOOKING — ₹3,200"
        )
    }
]

result = chat_with_model(zero_shot_messages)
print("🔵 Zero-Shot — Transaction Classification:\n")
print(result)

🔵 Zero-Shot — Transaction Classification:

| # | Transaction | Category |
|---|-------------|----------|
| 1 | UPI/SWIGGY/ORDER123 — ₹450 | Food & Dining |
| 2 | NEFT/HDFC HOME LOAN EMI — ₹23,000 | EMI/Loan Repayment |
| 3 | POS/BIGBASKET SUPERMART — ₹1,820 | Groceries |
| 4 | ATM/ICICI BANK/MG ROAD — ₹5,000 | ATM Withdrawal |
| 5 | IMPS/IRCTC TICKET BOOKING — ₹3,200 | Travel |


In [23]:
one_shot_messages = [
    {
        "role": "system",
        "content": (
            "You are a fraud detection alert system for a retail bank. "
            "Generate concise, professional SMS alerts for suspicious transactions. "
            "Keep messages under 160 characters and include: amount, merchant, last 4 card digits, and a helpline number."
        )
    },
    # One-shot example
    {
        "role": "user",
        "content": (
            "Generate a fraud alert SMS for:\n"
            "- Card: ending 4821\n"
            "- Amount: ₹12,500\n"
            "- Merchant: Amazon.in\n"
            "- Location: Mumbai\n"
            "- Helpline: 1800-XXX-XXXX"
        )
    },
    {
        "role": "assistant",
        "content": (
            "ALERT: Suspicious txn of ₹12,500 at Amazon.in on card **4821. "
            "If not you, call 1800-XXX-XXXX immediately to block. -YourBank"
        )
    },
    # Actual query
    {
        "role": "user",
        "content": (
            "Generate a fraud alert SMS for:\n"
            "- Card: ending 9034\n"
            "- Amount: ₹87,000\n"
            "- Merchant: Luxury Jewellers, Dubai\n"
            "- Location: International (UAE)\n"
            "- Helpline: 1800-XXX-XXXX"
        )
    }
]

result = chat_with_model(one_shot_messages)
print("🟡 One-Shot — Fraud Alert SMS:\n")
print(result)

🟡 One-Shot — Fraud Alert SMS:

ALERT: Suspicious txn of ₹87,000 at Luxury Jewellers, Dubai (International) on card **9034. If not you, call 1800-XXX-XXXX to block. -YourBank


In [24]:
few_shot_messages = [
    {
        "role": "system",
        "content": (
            "You are a senior credit risk analyst at a commercial bank. "
            "Evaluate loan applications and return a JSON object with:\n"
            "  - risk_level: Low / Medium / High\n"
            "  - credit_score_band: Excellent / Good / Fair / Poor\n"
            "  - debt_to_income: calculated as (monthly_obligations / monthly_income) * 100\n"
            "  - recommendation: Approve / Approve with Conditions / Reject\n"
            "  - reason: one concise sentence explaining the decision"
        )
    },
    # Example 1 — Low Risk
    {
        "role": "user",
        "content": (
            "Applicant: Priya Sharma\n"
            "Loan Requested: ₹15,00,000 (Home Loan)\n"
            "Monthly Income: ₹1,20,000\n"
            "Existing Monthly Obligations: ₹12,000\n"
            "Credit Score: 810\n"
            "Employment: Salaried — TCS (8 years)\n"
            "Collateral: Property worth ₹28,00,000"
        )
    },
    {
        "role": "assistant",
        "content": (
            '{\n'
            '  "risk_level": "Low",\n'
            '  "credit_score_band": "Excellent",\n'
            '  "debt_to_income": "10%",\n'
            '  "recommendation": "Approve",\n'
            '  "reason": "Strong credit score, stable long-term employment, and high-value collateral make this a low-risk application."\n'
            '}'
        )
    },
    # Example 2 — Medium Risk
    {
        "role": "user",
        "content": (
            "Applicant: Rajan Mehta\n"
            "Loan Requested: ₹8,00,000 (Personal Loan)\n"
            "Monthly Income: ₹55,000\n"
            "Existing Monthly Obligations: ₹18,000\n"
            "Credit Score: 680\n"
            "Employment: Self-Employed (3 years)\n"
            "Collateral: None"
        )
    },
    {
        "role": "assistant",
        "content": (
            '{\n'
            '  "risk_level": "Medium",\n'
            '  "credit_score_band": "Fair",\n'
            '  "debt_to_income": "33%",\n'
            '  "recommendation": "Approve with Conditions",\n'
            '  "reason": "Moderate credit score and no collateral increase risk; recommend approval at a reduced loan amount with a co-applicant or guarantor."\n'
            '}'
        )
    },
    # Example 3 — High Risk
    {
        "role": "user",
        "content": (
            "Applicant: Anil Desai\n"
            "Loan Requested: ₹12,00,000 (Business Loan)\n"
            "Monthly Income: ₹40,000\n"
            "Existing Monthly Obligations: ₹32,000\n"
            "Credit Score: 530\n"
            "Employment: Self-Employed (1 year)\n"
            "Collateral: None\n"
            "Previous Defaults: 1 (2022)"
        )
    },
    {
        "role": "assistant",
        "content": (
            '{\n'
            '  "risk_level": "High",\n'
            '  "credit_score_band": "Poor",\n'
            '  "debt_to_income": "80%",\n'
            '  "recommendation": "Reject",\n'
            '  "reason": "Critically high debt-to-income ratio, poor credit score, prior default history, and no collateral present an unacceptable credit risk."\n'
            '}'
        )
    },
    # Actual query
    {
        "role": "user",
        "content": (
            "Applicant: Sneha Iyer\n"
            "Loan Requested: ₹5,00,000 (Education Loan)\n"
            "Monthly Income: ₹70,000\n"
            "Existing Monthly Obligations: ₹22,000\n"
            "Credit Score: 725\n"
            "Employment: Salaried — Infosys (2 years)\n"
            "Collateral: None\n"
            "Co-Applicant: Father (retired govt. employee, pension ₹30,000/month)"
        )
    }
]

result = chat_with_model(few_shot_messages)
print("🟢 Few-Shot — Loan Risk Assessment:\n")
print(result)

🟢 Few-Shot — Loan Risk Assessment:

{
  "risk_level": "Medium",
  "credit_score_band": "Good",
  "debt_to_income": "31.43%",
  "recommendation": "Approve with Conditions",
  "reason": "Good credit score and stable salaried income support approval, but no collateral and only moderate income surplus warrant a co-applicant guarantee and standard education loan conditions."
}


In [25]:
complaint_messages = [
    {
        "role": "system",
        "content": (
            "You are a banking customer service router. "
            "Classify each complaint and return JSON with: "
            "department (Cards / Loans / NetBanking / Accounts / Fraud / KYC), "
            "priority (High / Medium / Low), and a one-line suggested_action."
        )
    },
    # Example 1
    {"role": "user",      "content": "My credit card was charged twice for the same order at Zomato yesterday."},
    {"role": "assistant", "content": '{"department": "Cards", "priority": "High", "suggested_action": "Initiate chargeback process and credit provisional refund within 24 hours."}'},
    # Example 2
    {"role": "user",      "content": "I am unable to log in to net banking since this morning; it keeps saying invalid credentials."},
    {"role": "assistant", "content": '{"department": "NetBanking", "priority": "Medium", "suggested_action": "Reset customer credentials and check for session lock in the access management system."}'},
    # Example 3
    {"role": "user",      "content": "I received an OTP I did not request. I think someone is trying to access my account."},
    {"role": "assistant", "content": '{"department": "Fraud", "priority": "High", "suggested_action": "Temporarily freeze account, invalidate active sessions, and escalate to the fraud investigation team immediately."}'},
    # Actual query
    {"role": "user",      "content": "My home loan EMI was deducted twice this month and my account balance is now negative."}
]

result = chat_with_model(complaint_messages)
print("🏦 Complaint Router Result:\n")
print(result)

🏦 Complaint Router Result:

{"department": "Loans", "priority": "High", "suggested_action": "Verify duplicate EMI debit, reverse excess amount, and waive any negative balance charges."}


In [26]:
complaint = "There is an unauthorized international transaction of ₹1,10,000 on my debit card."
print(f"Customer Complaint: '{complaint}'\n")
print("=" * 65)

# ── Zero-Shot ──────────────────────────────────────────────────────────
zs = chat_with_model([
    {"role": "system", "content": "You are a banking support agent. Classify and respond to customer complaints."},
    {"role": "user",   "content": complaint}
])
print("\n🔵 Zero-Shot:\n")
print(zs)

# ── One-Shot ───────────────────────────────────────────────────────────
os_ = chat_with_model([
    {"role": "system",    "content": "You are a banking support router. Return JSON: department, priority, suggested_action."},
    {"role": "user",      "content": "My credit card was charged twice for the same Zomato order."},
    {"role": "assistant", "content": '{"department": "Cards", "priority": "High", "suggested_action": "Initiate chargeback and issue provisional refund."}'},
    {"role": "user",      "content": complaint}
])
print("\n🟡 One-Shot:\n")
print(os_)

# ── Few-Shot ───────────────────────────────────────────────────────────
fs = chat_with_model([
    {"role": "system",    "content": "You are a banking support router. Return JSON: department, priority, suggested_action, and estimated_resolution_time."},
    {"role": "user",      "content": "My credit card was charged twice for the same Zomato order."},
    {"role": "assistant", "content": '{"department": "Cards", "priority": "High", "suggested_action": "Initiate chargeback and provisional refund.", "estimated_resolution_time": "3-5 business days"}'},
    {"role": "user",      "content": "I cannot log in to net banking since morning."},
    {"role": "assistant", "content": '{"department": "NetBanking", "priority": "Medium", "suggested_action": "Reset credentials and clear session locks.", "estimated_resolution_time": "Same day"}'},
    {"role": "user",      "content": "Someone transferred all my savings without my knowledge."},
    {"role": "assistant", "content": '{"department": "Fraud", "priority": "Critical", "suggested_action": "Freeze account, reverse transaction, escalate to fraud team.", "estimated_resolution_time": "Immediate — within 1 hour"}'},
    {"role": "user",      "content": complaint}
])
print("\n🟢 Few-Shot:\n")
print(fs)

print("\n" + "=" * 65)
print("Observation: Few-shot produces the most structured and actionable output.")

Customer Complaint: 'There is an unauthorized international transaction of ₹1,10,000 on my debit card.'


🔵 Zero-Shot:

**Classification:** Unauthorized/fraudulent debit card transaction — International, high-value (₹1,10,000).  
**Priority:** Critical / Fraud report  
**Type:** Unauthorized transaction dispute / chargeback

**Immediate response:**
I’m sorry this happened. Please treat this as urgent.

1. **Block your debit card immediately** if not already done — use the bank’s official app, net banking, or 24x7 fraud helpline.
2. **Do not share** your OTP, PIN, CVV, password, or full card number with anyone.
3. **Report the transaction to the bank** and ask for it to be registered as an **unauthorized/fraudulent transaction dispute**.
4. If your card was lost/stolen or your credentials were compromised, also file a complaint at **cybercrime.gov.in** or call **1930**.
5. Keep all SMS/email alerts and note the bank complaint/reference number.

**Safe details needed to raise the dispute